In [0]:
%sql

USE CATALOG bootcamp;

show schemas

In [0]:
%sql
create schema bronze

In [0]:
%sql

create volume landing.archivos
comment "Volume to storage bootcamp data";
show volumes in bootcamp.landing;

In [0]:
%sql

select * from read_files(
    '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
    format=>'csv',
    header=>true
)

In [0]:
%sql
CREATE TABLE bronze.properties_bronze
    select * from read_files(
        '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
        format=>'csv',
        header=>true
    )
    where url LIKE 'https%'

In [0]:
%sql

use bronze;
DESCRIBE properties_bronze

In [0]:
%sql

use samples.nyctaxi;

describe trips

In [0]:
%sql

with zonas as (
    select pickup_zip,count(*) as amount_trips, round(avg(fare_amount),2) as avg_fare, round(avg(trip_distance),2) as avg_distance
    from trips
     WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
    group by pickup_zip
)

select pickup_zip, amount_trips,avg_fare,  avg_distance from zonas
where amount_trips > 100


In [0]:
%sql

with avg_hora as (
    select hour(tpep_pickup_datetime) as hour,count(*) as Total_trips, round(avg(fare_amount),2) as hour_avg_fare
    from trips
    where fare_amount > 0
   
    group by hour(tpep_pickup_datetime)
),

general_avg as (

    select round(avg(fare_amount),2) as general_fare from trips
    where fare_amount > 0
   
)

select 
h.hour
,h.Total_trips
,h.hour_avg_fare
,g.general_fare
,round(((h.hour_avg_fare - g.general_fare)),2) as difference
,round(((h.hour_avg_fare - g.general_fare)/g.general_fare)*100,2) as difference_percentage 
from avg_hora h
cross join general_avg g
order by h.hour



In [0]:
%sql

with valid_trips as (
    select *
    from trips
    where fare_amount > 0
    and trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
)

select 
count(*) as total_trips
,min(fare_amount) as min_fare
,max(fare_amount) as max_fare
,round (avg(fare_amount),2) as avg_fare
,round(median(fare_amount),2) as median_fare
,min(trip_distance) as min_Distance
,max(trip_distance) as max_Distance
,round(avg(trip_distance),2) as avg_Distance
,median(trip_distance) as median_Distance

from valid_trips
    




In [0]:
%sql


select row_number()over(order by fare_amount desc) as ranking
,tpep_pickup_datetime,
round(trip_distance,2)  as trip_distance,
round(fare_amount,2) as fare_amount
from trips
where fare_amount > 0
order by ranking
limit 20

In [0]:
%sql

with zone_trips as (
select pickup_zip,count(*) as amount_trips
from trips
group by pickup_zip
)

select pickup_zip,amount_trips,
row_number() over(order by amount_trips desc) as row_number,
rank() over(order by amount_trips desc) as rank,
dense_rank() over(order by amount_trips desc) as dense_rank

from zone_trips  

In [0]:
%sql
SELECT 
  ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS ranking,
  tpep_pickup_datetime AS fecha,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20;

In [0]:
%sql

select 
tpep_pickup_datetime,
pickup_zip,
trip_distance,
fare_amount,
round(avg(fare_amount) over (),2) as avg_fare,
round((fare_amount-avg_fare),2) as difference,
round((fare_amount-avg_fare)/avg_fare*100,2) as difference_percentage
from trips
where fare_amount > 0
ORDER BY tpep_pickup_datetime DESC
LIMIT 20;
   



In [0]:
%sql

select 
tpep_pickup_datetime,
pickup_zip,
trip_distance,
fare_amount,
round(avg(fare_amount) over (partition by pickup_zip),2) as avg_fare_per_zone,
round((fare_amount-avg_fare_per_zone),2) as difference,
round((fare_amount-avg_fare_per_zone)/avg_fare_per_zone*100,2) as difference_percentage
from trips
WHERE fare_amount > 0
  AND pickup_zip IS NOT NULL
ORDER BY pickup_zip, fare_amount DESC
LIMIT 50;

In [0]:
%sql

select
tpep_pickup_datetime,
fare_amount,
lag(fare_amount,1) over (order by tpep_pickup_datetime) as previous_fare,
ROUND(fare_amount - LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime), 2) AS diferencia_con_anterior
from trips
where fare_amount > 0
order by tpep_pickup_datetime
limit 20


In [0]:
%sql

select
tpep_pickup_datetime,fare_amount,
ROUND(SUM(fare_amount) OVER (
    ORDER BY tpep_pickup_datetime 
  ), 2) AS total_acumulado
from trips
where fare_amount>0
order by tpep_pickup_datetime
limit 20

In [0]:
%sql

select
tpep_pickup_datetime,fare_amount,
sum(fare_amount) over(order by tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND
CURRENT ROW) as running_total
from trips
where fare_amount>0
order by tpep_pickup_datetime
limit 20

In [0]:
%sql

with valid_trips as (
    select *
    from trips
    where fare_amount > 0
    and trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
),

zone_values as(
select 
pickup_zip
,count(*) as total_trips
,min(fare_amount)  as min_fare_zone
,max(fare_amount)    as max_fare_zone
,round(avg(fare_amount),2)  as avg_fare_zone
from valid_trips
group by pickup_zip
having count(*) >= 50
)

select *,
row_number() over (order by avg_fare_zone desc) as ranking
from zone_values
order by ranking
limit 10


In [0]:
%sql
use bootcamp.bronze

In [0]:
%sql

select count(*) as total_registers from bootcamp.bronze.properties_bronze

In [0]:
%sql

describe properties_bronze

In [0]:
%sql
select id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales,
antiguedad, estado, zona from properties_bronze
limit 10

In [0]:
%sql
select 
count(*)-count(id) as total_id 
,count(*)-count(ubicacion) as total_ubicacion 
,count(*)-count(precio) as total_precio
,count(*)-count(expensas) as total_expensas
,count(*)-count(tipo_de_operacion) as total_Operacion 
,count(*)-count(moneda) as total_Moneda
,count(*)-count(ambientes) as total_ambientes
,count(*)-count(metros_cuadrados_totales) as total_Metros_cuadrados
,count(*)-count(antiguedad) as total_antiguedad
,count(*)-count(estado) as total_estado
,count(*)-count(zona) as total_zona 
 from properties_bronze

In [0]:
%sql
with totales as (

    select count(*) as total from properties_bronze
),

nulos as(
select

round((t.total-count(precio))*100/t.total,2) as percentage_nulls_precio, 
round((t.total-count(expensas))*100/t.total,2) as percentage_nulls_expensas, 
round((t.total-count(ambientes))*100/t.total,2) as percentage_nulls_ambientes, 
round((t.total-count(metros_cuadrados_totales))*100/t.total,2) as percentage_nulls_metros2,
round((t.total-count(metros_cuadrados_cubiertos))*100/t.total,2) as percentage_nulls_cubiertos, 
round((t.total-count(orientacion_cardinal))*100/t.total,2) as percentage_nulls_cardinal,
round((t.total-count(antiguedad))*100/t.total,2) as percentage_nulls_antiguedad

from properties_bronze 
cross join totales t
group by t.total)

SELECT columna, porcentaje_nulos
FROM nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        percentage_nulls_precio
        ,percentage_nulls_expensas
        ,percentage_nulls_ambientes
        ,percentage_nulls_metros2
        ,percentage_nulls_cubiertos
        ,percentage_nulls_cardinal
        ,percentage_nulls_antiguedad
       
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC


In [0]:
%sql

with tabla as(
select tipo_de_operacion, count(*) as total_tipo, sum(count(*)) over() as total_operaciones
from properties_bronze
group by tipo_de_operacion
order by total_operaciones desc)

select tipo_de_operacion,total_tipo, round(total_tipo*100/total_operaciones,2) as porcentaje from tabla
order by total_tipo desc

In [0]:
%sql

select moneda,count(*) as total_moneda, round(count(*)/sum(count(*))over()*100,2) as porcentaje from properties_bronze
group by moneda
order by total_moneda desc

In [0]:
%sql

select ambientes,count(*) as total_ambientes, round(count(*)/sum(count(*))over()*100,2) as porcentaje from properties_bronze
group by ambientes
order by ambientes

In [0]:
%sql

select zona, count(*) as total_zona, round(count(*)/sum(count(*))over()*100,2) as porcentaje from properties_bronze
group by zona
order by total_zona desc
limit 15

In [0]:
%sql

select estado, count(*) as total_estado, round(count(*)/sum(count(*))over()*100,2) as porcentaje from properties_bronze
group by estado
order by total_estado desc

In [0]:
%sql

create or replace temporary view propiedades_clean as
select
case 
    when precio RLIKE '^[^a-zA-Z]+$' then precio::double
    else null
end as precio,
moneda,
case 
    when expensas RLIKE '^[^a-zA-Z]+$' then Expensas::double
    else null
end as Expensas,
case 
    when ambientes RLIKE '^[^a-zA-Z]+$' then ambientes::double
    else null
end as ambientes,
case 
    when metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' then metros_cuadrados_totales::double
    else null
end as metros_cuadrados_totales,
case 
    when metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' then metros_cuadrados_cubiertos::double
    else null
end as metros_cuadrados_cubiertos,
case 
    when antiguedad RLIKE '^[^a-zA-Z]+$' then antiguedad::double
    else null
end as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from properties_bronze




In [0]:
%sql

select moneda,tipo_de_operacion,count(*) as total_operaciones, min(precio) as minimo, max(precio) as maximo, round(percentile(precio,0.25),2) as pc25,round(avg(precio),2) as promedio, round(percentile(precio,0.75),2) as pc75 from propiedades_clean 
where precio >0
group by moneda,tipo_de_operacion
order by total_operaciones desc
    

  



In [0]:
%sql

select 'Metros Totales' as Tipo_de_medida,count(*) as total,round(percentile(metros_cuadrados_totales,0.25),2) as pc25_Totales,round(percentile(metros_cuadrados_totales,0.5),2) as mediana_totales, round(percentile(metros_cuadrados_totales,0.75),2) as pc75_totales from propiedades_clean 
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

union all

select 'Metros Cubiertos' as Tipo_de_medida,count(*) as total,round(percentile(metros_cuadrados_cubiertos,0.25),2) as pc25_Cubiertos,round(percentile(metros_cuadrados_Cubiertos,0.5),2) as mediana_Cubiertos, round(percentile(metros_cuadrados_Cubiertos,0.75),2) as pc75_Cubiertos from propiedades_clean 
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;

In [0]:
%sql

select antiguedad,count(*) as total from propiedades_clean 
where antiguedad is not null and antiguedad >0
group by antiguedad
order by total desc
limit 20

In [0]:
%sql

select count(*) as total, min(antiguedad) as min_antiguedad ,max(antiguedad) as max_antiguedad,round(avg(antiguedad),2) as average, percentile(antiguedad,0.5) as median from propiedades_clean 
where antiguedad is not null and antiguedad >=0 and antiguedad !=999
order by total desc

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
%sql

with total AS (
    SELECT COUNT(*) as n 
    FROM propiedades_clean_2
 ),
problemas as( 

    select
    count(case when precio is null or precio::float<=0 then 1 end) as invalid_prices,
    count(case when Metros_cuadrados_totales is null or Metros_cuadrados_totales::float<=0 then 1 end) as invalid_M2,
    count(case when  Antiguedad=999 or antiguedad='999' then 1 end) as invalid_antiguedad,
    count(case when ambientes is null or ambientes=0 then 1 end) as invalid_ambientes,
    count(case when moneda is null or moneda='' then 1 end) as invalid_moneda,
    count(case when tipo_de_operacion is null or tipo_de_operacion='' then 1 end) as invalid_operacion
  
  from propiedades_clean_2

)

select t.n as total_registros,
ROUND(p.invalid_prices * 100.0 / t.n, 2) as pct_precio_invalido,
    p.invalid_M2,
    ROUND(p.invalid_M2 * 100.0 / t.n, 2) as pct_m2_invalido,
    p.invalid_antiguedad,
    ROUND(p.invalid_antiguedad * 100.0 / t.n, 2) as pct_antiguedad_999,
    p.invalid_ambientes,
    ROUND(p.invalid_ambientes * 100.0 / t.n, 2) as pct_ambientes_invalido,
    p.invalid_moneda,
    ROUND(p.invalid_moneda * 100.0 / t.n, 2) as pct_ambientes_invalido,
    p.invalid_operacion,
    ROUND(p.invalid_operacion * 100.0 / t.n, 2) as pct_ambientes_invalido

FROM total t, problemas p;

In [0]:
%sql
with duplicados as (                   
select url,precio,count(*) as duplicados from properties_bronze
group by url,precio 
having count(*)>1)

select count(*) as grupo_duplicado,
sum(duplicados) as total_duplicados,
sum(duplicados-1) as registros_extra
from duplicados

In [0]:
%sql
with duplicados as (                   
select url,precio,count(*) as duplicados from properties_bronze
group by url,precio 
having count(*)>1)

select *
from duplicados
order by duplicados desc
limit 10

In [0]:
%sql

with outliers as(
select moneda,
percentile(precio,0.01) as p01,
percentile(precio,0.99) as p99
from propiedades_clean_2
where precio>0
group by moneda
)

    select p.moneda,
    "Muy Bajo" as Tipo_outlier,
    count(*) as cantidad
    from propiedades_clean_2 p
        join outliers o
    on p.moneda=o.moneda
    where precio < p01 and p.precio>0
    group by p.moneda

union all
    select p.moneda,
    "Muy Alto" as Tipo_outlier,
    count(*) as cantidad
    from propiedades_clean_2 p
    join outliers o
    on p.moneda=o.moneda
    where precio > p99 and p.precio>0
     group by p.moneda


In [0]:
%sql

select zona,
 round(avg(precio),2) as precio_promedio,
 count(*) as cantidad_propiedades,
 row_number() over(order by round(avg(precio),2) desc) as ranking
from propiedades_clean
where 
zona is not null
and precio is not null and precio>0
group by zona
order by ranking

In [0]:
%sql
create or replace temp view bronze_EDA as (
SELECT 
    edl.id,
    edl.ubicacion,
    CASE 
        WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.precio::float BETWEEN -2147483648 AND 2147483648 THEN edl.precio::float
        ELSE NULL
    END AS precio,
    CASE 
        WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
        ELSE NULL
    END AS numero,
    edl.calle,
    CASE
        WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.expensas::float BETWEEN -20000000 AND 20000000 THEN edl.expensas::float
        ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
        WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%mxn%' THEN 'MXN'
        WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
        WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
        ELSE edl.moneda
    END AS moneda,
    CASE
        WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.ambientes::float
    END AS ambientes,
    CASE 
        WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE 
        WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
        WHEN edl.piso IS NULL THEN NULL
        WHEN edl.piso = 'NaN' THEN NULL
        ELSE edl.piso::float
    END AS piso,
    CASE 
        WHEN edl.cochera = 'tiene' THEN 1
        ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    edl.fecha,
    edl.hora
FROM propiedades_clean_2 edl
);

In [0]:
%sql
WITH precio_por_zona AS (
    SELECT 
        zona,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio_zona
    FROM bronze_EDA
    WHERE precio > 0 
      AND zona IS NOT NULL
    GROUP BY zona

)
SELECT 
    zona,
    cantidad as propiedades,
    precio_promedio_zona,
    ROUND(AVG(precio_promedio_zona) OVER (), 2) as precio_promedio_general,
    ROUND(precio_promedio_zona - AVG(precio_promedio_zona) OVER (), 2) as diferencia,
    ROUND((precio_promedio_zona - AVG(precio_promedio_zona) OVER ()) * 100.0 / AVG(precio_promedio_zona) OVER (), 2) as porcentaje_diferencia
FROM precio_por_zona
ORDER BY precio_promedio_zona DESC
LIMIT 15

In [0]:
%sql

select distinct month(fecha) as mes,
count(*) as total,
round(avg(precio),2) as precio_promedio
from bronze_EDA
where moneda ='ARS' and tipo_de_operacion ='alquiler'
group by month(fecha)